In [31]:
import pandas as pd
from io import BytesIO, StringIO
from zipfile import ZipFile
import glob
import csv
import numpy as np


In [3]:
path = "/Volumes/2/casa/data/*.csv"
for filename in glob.glob(path):
    reader = pd.read_csv(filename, sep = None, iterator = True, engine = 'python')
    inferred_separator = reader._engine.data.dialect.delimiter
    print(inferred_separator, filename, 'TIMESTAMP' in pd.read_csv(filename, nrows=0, sep=inferred_separator).columns.tolist())

, /Volumes/2/casa/data/salesstock$vw-04-06_2023.csv True
; /Volumes/2/casa/data/salesstock$vw-10-12_2022.csv True
; /Volumes/2/casa/data/salesstock$vw-09_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-10-12_2021.csv True
; /Volumes/2/casa/data/salesstock$vw-03_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-04_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-12_2018.csv True
; /Volumes/2/casa/data/salesstock$vw-12_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-07-08_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-11-12_2020.csv True
, /Volumes/2/casa/data/salesstock$vw-05-06_2019.csv True
; /Volumes/2/casa/data/salesstock$vw-07-09_2023.csv True
; /Volumes/2/casa/data/salesstock$vw-07-09_2022.csv True
, /Volumes/2/casa/data/salesstock$vw-04-05_2021.csv True
, /Volumes/2/casa/data/salesstock$vw-04-05_2020.csv True
; /Volumes/2/casa/data/salesstock$vw-07-09_2021.csv True
; /Volumes/2/casa/data/salesstock$vw-11_2018.csv True
; /Volumes/2/casa/data/salesstock$vw-10_2018.csv 

In [3]:
path1 = "/Volumes/2/casa/timestamp/*.csv"
path2 = "/Volumes/2/casa/data/*.csv"

path_list1 = [s.split("/")[-1] for s in glob.glob(path1)]
path_list2 = [s.split("/")[-1] for s in glob.glob(path2)]

for i, filename in enumerate(path_list1):
    print(i+1, filename, end='\n')

print(set(path_list1) <= (set(path_list2)))

1 salesstock$vw-04-06_2023.csv
2 salesstock$vw-10-12_2022.csv
3 salesstock$vw-09_2019.csv
4 salesstock$vw-10-12_2021.csv
5 salesstock$vw-03_2019.csv
6 salesstock$vw-04_2019.csv
7 salesstock$vw-12_2018.csv
8 salesstock$vw-12_2019.csv
9 salesstock$vw-07-08_2019.csv
10 salesstock$vw-11-12_2020.csv
11 salesstock$vw-07-09_2023.csv
12 salesstock$vw-07-09_2022.csv
13 salesstock$vw-04-05_2021.csv
14 salesstock$vw-07-09_2021.csv
15 salesstock$vw-11_2018.csv
16 salesstock$vw-10_2018.csv
17 salesstock$vw-10_2019.csv
18 salesstock$vw-11_2019.csv
19 salesstock$vw-06-07_2020.csv
20 salesstock$vw-06_2021.csv
21 salesstock$vw-01-03_2021.csv
22 salesstock$vw-08-09_2020.csv
23 salesstock$vw-01-03_2020.csv
24 salesstock$vw-10_2020.csv
25 salesstock$vw-01-03_2023.csv
True


In [4]:
folder1 = "/Volumes/2/casa/timestamp/"
folder2 = "/Volumes/2/casa/data/"

path1 = folder1 + "*.csv"
path2 = folder2 + "*.csv"

path_list1 = [s.split("/")[-1] for s in glob.glob(path1)]
path_list2 = [s.split("/")[-1] for s in glob.glob(path2)]

for i, path in enumerate(path_list2):
    reader2 = pd.read_csv(folder2 + path, sep = None, iterator = True, engine = 'python')
    inferred_separator2 = reader2._engine.data.dialect.delimiter
    if 'TIMESTAMP' not in pd.read_csv(folder2 + path, nrows=0, sep=inferred_separator2).columns.tolist():
        print(i, path)

7 salesstock$vw-12_2019.csv
9 salesstock$vw-11-12_2020.csv
12 salesstock$vw-07-09_2022.csv
15 salesstock$vw-07-09_2021.csv
17 salesstock$vw-10_2018.csv
20 salesstock$vw-06-07_2020.csv
21 salesstock$vw-06_2021.csv
23 salesstock$vw-01-03_2021.csv
24 salesstock$vw-08-09_2020.csv
25 salesstock$vw-01-03_2020.csv


In [3]:
folder1 = "/Volumes/2/casa/timestamp/"
folder2 = "/Volumes/2/casa/data/"

path1 = folder1 + "*.csv"
path2 = folder2 + "*.csv"

path_list1 = [s.split("/")[-1] for s in glob.glob(path1)]
path_list2 = [s.split("/")[-1] for s in glob.glob(path2)]

for i, path in enumerate(path_list2):
    reader2 = pd.read_csv(folder2 + path, sep = None, iterator = True, engine = 'python')
    inferred_separator2 = reader2._engine.data.dialect.delimiter
    if 'TIMESTAMP' not in pd.read_csv(folder2 + path, nrows=0, sep=inferred_separator2).columns.tolist():
        reader1 = pd.read_csv(folder1 + path, sep = None, iterator = True, engine = 'python')
        inferred_separator1 = reader1._engine.data.dialect.delimiter
        try:
            df1 = pd.read_csv(folder1 + path, low_memory=False, sep=inferred_separator1)
            df2 = pd.read_csv(folder2 + path, low_memory=False, sep=inferred_separator2)
        except Exception as e:
            print(e, i, path, end='\n')
            continue
        if df1['SALESSTOCKID'].equals(df2['SALESSTOCKID']):
            df2['TIMESTAMP'] = df1['TIMESTAMP']
            df2.to_csv(folder2 + path, index=False, quoting=csv.QUOTE_ALL, sep=inferred_separator2)
            print("EQ", i, path, end='\n')
        else:
            # df2.merge(df1, on=['SALESSTOCKID'], how='left').to_csv(folder2 + path, index=False)
            print("MERGE", i, path, end='\n')

EQ 8 salesstock$vw-04_2019.csv
EQ 18 salesstock$vw-07-09_2021.csv
EQ 20 salesstock$vw-10_2018.csv
EQ 25 salesstock$vw-08-09_2020.csv
EQ 26 salesstock$vw-01-03_2020.csv


In [26]:
import regex as re
re.sub("[^\d\,]", "", "1:")

'1'

In [10]:
folder_data = "/Volumes/2/casa/data/"
folder_daily = "/Volumes/2/casa/daily_usercount/"
file_list = [filename.split("/")[-1] for filename in glob.glob(folder_daily + "*.csv")]
header = ['TIMESTAMP', 'SECONDSSPENT', 'QUANTITY', 'VOLUME', 'WEIGHT', 'PRICE', 'TEAMNAME', 'USERNAME']

for i, filepath in enumerate(glob.glob(folder_data + "*.csv")):
    if filepath.split("/")[-1] not in file_list:
        try:
            print(filepath)
            reader = pd.read_csv(filepath, sep = None, iterator = True, engine = 'python')
            inferred_separator = reader._engine.data.dialect.delimiter

            df = pd.read_csv(filepath, usecols=header, low_memory=False, sep=inferred_separator)[header]
            df['VOLUME'] = df['VOLUME'].str.replace('[^\d\,]', '', regex=True)
            df['VOLUME'] = df['VOLUME'].str.replace('\s*', '0', regex=True)
            df['VOLUME'] = df['VOLUME'].str.replace(',', '.').astype(float)
            df['WEIGHT'] = df['WEIGHT'].str.replace('[^\d\,]', '', regex=True)
            df['WEIGHT'] = df['WEIGHT'].str.replace('\s*', '0', regex=True)
            df['WEIGHT'] = df['WEIGHT'].str.replace(',', '.', regex=True).astype(float)
            df['PRICE'] = df['PRICE'].str.replace('[^\d\,]', '', regex=True)
            df['PRICE'] = df['PRICE'].str.replace('\s*', '0', regex=True)
            df['PRICE'] = df['PRICE'].str.replace(',', '.').astype(float)

            # df['TIMESTAMP'] = df['TIMESTAMP'].str.strip()
            df['DATE'] = df['TIMESTAMP'].str.split(" ", expand=True)[0]
            df['DATE'] = df['DATE'].str.zfill(10)
            df = df.drop(['TIMESTAMP'], axis=1)
            df['DATE'] = pd.to_datetime(df['DATE'], format="%d/%m/%Y")
            (df.groupby(['DATE', 'TEAMNAME'])
                .agg({'USERNAME':'count', 'QUANTITY': 'sum', 'VOLUME': 'sum', 'WEIGHT': 'sum', 'PRICE': 'sum'})
                .reset_index()
                .rename(columns={'USERNAME': 'USERCOUNT'})
            )
            df.sort_values(by=['DATE'], inplace=True)
            df.to_csv(folder_daily + filepath.split("/")[-1], index=True, quoting=csv.QUOTE_ALL, sep=inferred_separator)
            print(i+1, filepath.split("/")[-1], end='\n')
        except Exception as e:
            print(i+1, filepath.split("/")[-1] + '\n', e)
            continue


/Volumes/2/casa/data/salesstockvw-10_2019.csv
1 salesstockvw-10_2019.csv
/Volumes/2/casa/data/salesstockvw-11_2019.csv
2 salesstockvw-11_2019.csv
/Volumes/2/casa/data/salesstock$vw-04-06_2023.csv
3 salesstock$vw-04-06_2023.csv
/Volumes/2/casa/data/salesstock$vw-10-12_2022.csv
4 salesstock$vw-10-12_2022.csv
/Volumes/2/casa/data/salesstock$vw-09_2019.csv
5 salesstock$vw-09_2019.csv
/Volumes/2/casa/data/salesstock$vw-10-12_2021.csv
6 salesstock$vw-10-12_2021.csv
/Volumes/2/casa/data/salesstock$vw-03_2019.csv
7 salesstock$vw-03_2019.csv
/Volumes/2/casa/data/salesstockvw-10_2020.csv
8 salesstockvw-10_2020.csv
/Volumes/2/casa/data/salesstock$vw-04_2019.csv
9 salesstock$vw-04_2019.csv
/Volumes/2/casa/data/salesstock$vw-12_2018.csv
10 salesstock$vw-12_2018.csv
/Volumes/2/casa/data/salesstock$vw-12_2019.csv
11 salesstock$vw-12_2019.csv
/Volumes/2/casa/data/salesstock$vw-07-08_2019.csv
12 salesstock$vw-07-08_2019.csv
/Volumes/2/casa/data/salesstock$vw-11-12_2020.csv
13 salesstock$vw-11-12_2020.c

In [3]:
folder_clean = "/Volumes/2/clean/"
folder_daily = "/Volumes/2/casa/daily_usercount/"
df = pd.DataFrame()

for filepath in glob.glob(folder_daily + "*.csv"):
    reader = pd.read_csv(filepath, sep = None, iterator = True, engine = 'python')
    inferred_separator = reader._engine.data.dialect.delimiter
    df = pd.concat([df, pd.read_csv(filepath, sep=inferred_separator)])

df.sort_values(by=['DATE'], inplace=True)
df.to_csv(folder_clean + "daily_usercount.csv", index=False, quoting=csv.QUOTE_ALL)


In [4]:
folder_data = "/Volumes/2/casa/data/"
folder_daily = "/Volumes/2/casa/daily_usercount/"
header = ['TIMESTAMP', 'SECONDSSPENT', 'QUANTITY', 'VOLUME', 'WEIGHT', 'PRICE', 'TEAMNAME', 'USERNAME']

filepath = "/Volumes/2/casa/data/salesstock$vw-10-12_2021.csv"

try:
    print(filepath)
    reader = pd.read_csv(filepath, sep = None, iterator = True, engine = 'python')
    inferred_separator = reader._engine.data.dialect.delimiter

    df = pd.read_csv(filepath, usecols=header, low_memory=False, sep=inferred_separator)[header]
    df['VOLUME'] = df['VOLUME'].str.replace('[^\d\,]', '', regex=True)
    df['VOLUME'] = df['VOLUME'].str.replace('^\s*', '0', regex=True)
    df['VOLUME'] = df['VOLUME'].str.replace(',', '.').astype(float)
    df['WEIGHT'] = df['WEIGHT'].str.replace('[^\d\,]', '', regex=True)
    df['WEIGHT'] = df['WEIGHT'].str.replace('^\s*', '0', regex=True)
    df['WEIGHT'] = df['WEIGHT'].str.replace(',', '.').astype(float)
    df['PRICE'] = df['PRICE'].str.replace('[^\d\,]', '', regex=True)
    df['PRICE'] = df['PRICE'].str.replace('^\s*', '0', regex=True)
    df['PRICE'] = df['PRICE'].str.replace(',', '.').astype(float)

    # df['TIMESTAMP'] = df['TIMESTAMP'].str.strip()
    # df[['DATE', 'TIME']] = df['TIMESTAMP'].str.split(" ", expand=True)
    df['DATE'] = df['TIMESTAMP'].str.split(" ", expand=True)[0]
    # df['TIME'] = df['TIMESTAMP'].str.split(" ", expand=True)[1]
    df['DATE'] = df['DATE'].str.zfill(10)
    df = df.drop(['TIMESTAMP'], axis=1)
    df['DATE'] = pd.to_datetime(df['DATE'], format="%d/%m/%Y")
    df = df.groupby(['DATE', 'TEAMNAME', 'USERNAME']).sum()
    df.sort_values(by=['DATE', 'USERNAME'], inplace=True)
    df.to_csv(folder_daily + filepath.split("/")[-1], index=True, quoting=csv.QUOTE_ALL, sep=inferred_separator)
    print(filepath.split("/")[-1], end='\n')
except Exception as e:
    print(filepath.split("/")[-1] + '\n', e)

/Volumes/2/casa/data/salesstock$vw-10-12_2021.csv
salesstock$vw-10-12_2021.csv


In [26]:
filepath = "/Volumes/2/casa/data/salesstock$vw-01-03_2022.csv"
header = ['TIMESTAMP', 'SECONDSSPENT', 'QUANTITY', 'VOLUME', 'WEIGHT', 'PRICE', 'TEAMNAME', 'USERNAME']

reader = pd.read_csv(filepath, sep = None, iterator = True, engine = 'python')
inferred_separator = reader._engine.data.dialect.delimiter
df = pd.read_csv(filepath, usecols=header, low_memory=False, sep=inferred_separator)[header]
df['PRICE'] = df['PRICE'].str.replace('[^\d\,]', '', regex=True)
df['PRICE'] = df['PRICE'].str.replace('^\s*', '0', regex=True)
df['PRICE'] = df['PRICE'].str.replace(',', '.').astype(float)
sum(df['PRICE'])

35223905.02944706

In [33]:
np.sum(list(df['PRICE']))

35223905.03

In [30]:
list(df['PRICE'])[:1000]

[250.0,
 139.0,
 399.0,
 299.0,
 24.95,
 24.95,
 24.95,
 2.95,
 21.95,
 4.95,
 9.95,
 3.95,
 5.95,
 8.95,
 2.95,
 5.95,
 29.95,
 39.95,
 13.0,
 49.95,
 21.95,
 14.95,
 39.95,
 89.0,
 89.0,
 19.95,
 25.0,
 17.95,
 21.95,
 15.95,
 399.0,
 21.95,
 139.0,
 199.0,
 199.0,
 279.0,
 8.5,
 4.5,
 3.95,
 3.95,
 3.95,
 3.95,
 3.95,
 3.95,
 3.95,
 49.95,
 39.95,
 9.95,
 35.95,
 21.95,
 9.95,
 21.95,
 50.0,
 19.95,
 29.95,
 39.95,
 39.95,
 27.95,
 59.0,
 27.95,
 39.95,
 7.95,
 7.95,
 7.95,
 14.95,
 14.95,
 14.95,
 24.95,
 37.95,
 4.95,
 39.95,
 6.95,
 6.95,
 35.0,
 45.0,
 12.95,
 15.95,
 35.0,
 35.0,
 15.95,
 4.95,
 55.0,
 55.0,
 24.95,
 55.0,
 55.0,
 29.95,
 2.95,
 119.0,
 119.0,
 49.95,
 11.95,
 11.95,
 49.95,
 29.95,
 399.0,
 1.95,
 1.95,
 1.95,
 1.95,
 2.75,
 2.75,
 39.95,
 14.95,
 99.0,
 4.5,
 5.5,
 8.5,
 49.95,
 45.0,
 29.95,
 2.75,
 14.95,
 11.95,
 11.95,
 11.95,
 19.95,
 11.95,
 9.95,
 79.0,
 39.0,
 9.95,
 79.0,
 44.95,
 6.95,
 36.95,
 4.95,
 4.95,
 21.95,
 89.0,
 22.95,
 11.95,
 3.5,
 3.95

In [21]:
df_try = df.copy()
(df_try.groupby(['DATE', 'TEAMNAME'])
    .agg({'USERNAME':'count', 'QUANTITY': 'sum', 'VOLUME': 'sum', 'WEIGHT': 'sum', 'PRICE': 'sum'})
    .reset_index()
    .rename(columns={'USERNAME': 'USERCOUNT'})
)

,DATE,TEAMNAME,USERCOUNT,QUANTITY,VOLUME,WEIGHT,PRICE
0,2021-10-01,DD,5,1139411,280.291453,2.577021e+07,4.957479e+08
1,2021-10-01,DP1,47,598318,172.145910,1.153363e+07,2.349545e+08
2,2021-10-01,DP2,48,519875,509.743444,1.732620e+07,2.885524e+08
3,2021-10-01,EXT2,1,6892,1.677010,1.328640e+05,5.363238e+06
4,2021-10-01,HD,1,4188,1.999181,7.076714e+04,3.149825e+06
...,...,...,...,...,...,...,...
416,2021-12-30,DP1,50,408050,302.090524,1.190130e+07,2.779821e+08
417,2021-12-30,DP2,47,584062,304.779086,1.712043e+07,3.113027e+08
418,2021-12-30,HD,1,810,21.769162,2.619701e+05,1.063808e+07
419,2021-12-30,HP1,1,17210,2.592927,3.833821e+05,7.657572e+06


In [ ]:
from datetime import datetime
datetime.strptime("3/12/2018 9:00:00", "%d/%m/%Y %H:%M:%S")


datetime.datetime(2018, 12, 3, 9, 0)

In [ ]:
df_full = pd.read_csv('/Volumes/2/data/timestamp/salesstock$vw-12_2018.csv', low_memory=False, sep=',')
df[['DATE', 'TIME']] = df['TIMESTAMP'].str.split(' ', n=1, expand=True)
df_full['TIMESTAMP'] = df_full['DATE'].str.zfill(10) + " " + df_full['TIME'].str.zfill(8)
df_full = df_full.drop(['DATE', 'TIME'], axis=1)
df_full['TIMESTAMP'] = pd.to_datetime(df_full['TIMESTAMP'], format="%d/%m/%Y %H:%M:%S")
df_full.sort_values(by='TIMESTAMP')


,TIMESTAMP,SALESSTOCKID
3686713,2018-12-01 02:00:00,235175327
3686714,2018-12-01 07:00:00,235175328
2093033,2018-12-03 06:00:00,235188032
2093107,2018-12-03 06:00:00,235177547
2093105,2018-12-03 06:00:00,235183217
...,...,...
4392322,2018-12-29 10:00:00,240470710
4392323,2018-12-29 10:00:00,240470707
3717163,2018-12-29 11:00:00,240474235
3717162,2018-12-29 19:00:00,240474236


In [ ]:
# # CHECK LATER !!! all the files with timestamp column

# path = 'salesstock$vw-04-06_2023.csv'

# reader1 = pd.read_csv(folder1 + path, sep = None, iterator = True, engine = 'python')
# inferred_separator1 = reader1._engine.data.dialect.delimiter
# reader2 = pd.read_csv(folder2 + path, sep = None, iterator = True, engine = 'python')
# inferred_separator2 = reader2._engine.data.dialect.delimiter

# df1 = pd.read_csv(folder1 + path, low_memory=False, sep=inferred_separator1)
# df2 = pd.read_csv(folder2 + path, low_memory=False, sep=inferred_separator2)
# print(len(df1), len(df2))
# # df2.merge(df1, on='SALESSTOCKID', how='left')

8515814 8515814


In [6]:
pd.options.display.max_columns = None
path = '/Volumes/2/casa/data/salesstock$vw-04-06_2023.csv'
df = pd.read_csv(path, low_memory=False, sep=',')
df.head()

,ARTICLEID,LOCATIONID,PICKLOCATIONID,TASKATTRIBUTEID,DSPWAVEID,ORIGINALQUANTITY,QUANTITY,VOLUME,WEIGHT,SECONDSSPENT,TIMESTAMP,DONETIMESTAMP,ARTNR,ARTDESCR,LOCNAME,RES,USERNAME,TEAMNAME,RECIPIENT,RCPNAME,ORDTYPE,ORDARTNORMTIMEGROUP,TRIGGERMOMENT,PALNUMBER,CLAIMID,PICKCLAIMID,TASKID,PRICE,CONTTYPE,CTNAME,SALESSTOCKID
0,130561,286017,301724,399,181395.0,1,1,"0,01269","1,29",0,3/04/2023 8:00:00,3/04/2023 19:00:00,582057,CUBO WANDREK S/3,039053010A2,DREC-5,User316,DP1,1021071847,DUFAU Marie,WC,NORMD30,DSP,9140388,192871962,192871962,91210090.0,"29,95",503,503-Steunpallet:,480109379
1,136522,286017,300576,399,181395.0,1,1,"0,003861","1,53",0,3/04/2023 8:00:00,3/04/2023 19:00:00,676011,ACACIA STYLE SERVEERPLANK 60X2,039053010A2,DREC-5,User316,DP1,1026078489,Kolen Wendy,WC,NORMD30,DSP,9140388,192872003,192872003,91210090.0,"24,95",503,503-Steunpallet:,480109391
2,131913,282072,282072,451,182697.0,1,1,"0,438372","3,54",244,9/05/2023 9:00:00,9/05/2023 21:00:00,660436,ACAPULCO LOUNGE STOEL NATUREL,03606613700,DORPV65,User1896,DP1,1024030724,Rodriguez Hernandez Manuel,WP,NORMD65,PCK,9181459,193386797,193386797,91471068.0,59,503,503-Steunpallet:,483809020
3,131913,282072,282072,451,182697.0,1,1,"0,438372","3,54",0,9/05/2023 9:00:00,9/05/2023 21:00:00,660436,ACAPULCO LOUNGE STOEL NATUREL,03606613700,DORPV65,User1896,DP1,1024030724,Rodriguez Hernandez Manuel,WP,NORMD65,PCK,9181459,193386798,193386798,91471068.0,59,503,503-Steunpallet:,483809021
4,130703,305046,305046,389,181645.0,1,1,"0,00080625","0,379166666667",169,11/04/2023 9:00:00,11/04/2023 20:00:00,527044,MOON BOWL 49CL,039053001A3D1,D-LAADLOS07,User293,DP2,1021072312,Fleurant Daniel,WC,NORMD40,PCK,KRAT055,192972972,192972972,91255954.0,"3,5",EC1,EC1 bak e-com pick,480833847


In [7]:
df.head(100)

,ARTICLEID,LOCATIONID,PICKLOCATIONID,TASKATTRIBUTEID,DSPWAVEID,ORIGINALQUANTITY,QUANTITY,VOLUME,WEIGHT,SECONDSSPENT,TIMESTAMP,DONETIMESTAMP,ARTNR,ARTDESCR,LOCNAME,RES,USERNAME,TEAMNAME,RECIPIENT,RCPNAME,ORDTYPE,ORDARTNORMTIMEGROUP,TRIGGERMOMENT,PALNUMBER,CLAIMID,PICKCLAIMID,TASKID,PRICE,CONTTYPE,CTNAME,SALESSTOCKID
0,130561,286017,301724,399,181395.0,1,1,"0,01269","1,29",0,3/04/2023 8:00:00,3/04/2023 19:00:00,582057,CUBO WANDREK S/3,039053010A2,DREC-5,User316,DP1,1021071847,DUFAU Marie,WC,NORMD30,DSP,9140388,192871962,192871962,91210090.0,"29,95",503,503-Steunpallet:,480109379
1,136522,286017,300576,399,181395.0,1,1,"0,003861","1,53",0,3/04/2023 8:00:00,3/04/2023 19:00:00,676011,ACACIA STYLE SERVEERPLANK 60X2,039053010A2,DREC-5,User316,DP1,1026078489,Kolen Wendy,WC,NORMD30,DSP,9140388,192872003,192872003,91210090.0,"24,95",503,503-Steunpallet:,480109391
2,131913,282072,282072,451,182697.0,1,1,"0,438372","3,54",244,9/05/2023 9:00:00,9/05/2023 21:00:00,660436,ACAPULCO LOUNGE STOEL NATUREL,03606613700,DORPV65,User1896,DP1,1024030724,Rodriguez Hernandez Manuel,WP,NORMD65,PCK,9181459,193386797,193386797,91471068.0,59,503,503-Steunpallet:,483809020
3,131913,282072,282072,451,182697.0,1,1,"0,438372","3,54",0,9/05/2023 9:00:00,9/05/2023 21:00:00,660436,ACAPULCO LOUNGE STOEL NATUREL,03606613700,DORPV65,User1896,DP1,1024030724,Rodriguez Hernandez Manuel,WP,NORMD65,PCK,9181459,193386798,193386798,91471068.0,59,503,503-Steunpallet:,483809021
4,130703,305046,305046,389,181645.0,1,1,"0,00080625","0,379166666667",169,11/04/2023 9:00:00,11/04/2023 20:00:00,527044,MOON BOWL 49CL,039053001A3D1,D-LAADLOS07,User293,DP2,1021072312,Fleurant Daniel,WC,NORMD40,PCK,KRAT055,192972972,192972972,91255954.0,"3,5",EC1,EC1 bak e-com pick,480833847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,133581,301723,319406,390,183140.0,2,2,"0,001125","0,424",2,19/05/2023 11:00:00,19/05/2023 19:00:00,629895,EARTH MARL MOK 20CL,039051095A0,DREC-5,User1587,DP1,1026081478,Schouten Peter,WC,NORMD40,DSP,KRAT093,193545497,193545497,91561142.0,"3,29",EC1,EC1 bak e-com pick,484748120
96,139603,287620,287620,451,183178.0,2,2,"0,056056","10,15",111,19/05/2023 16:00:00,19/05/2023 21:00:00,652631,IMPERIAL BISTRO STOEL GEEL,039054075A0,DCONS08,User1428,DP1,1023010779,soares paulo,WP,NORMD60,PCK,9196212,193554189,193554189,91563661.0,"29,95",503,503-Steunpallet:,484779152
97,139603,287620,287620,451,183178.0,0,0,0,0,0,19/05/2023 16:00:00,19/05/2023 21:00:00,652631,IMPERIAL BISTRO STOEL GEEL,039054075A0,DCONS08,User1428,DP1,1023010779,soares paulo,WP,NORMD60,PCK,9196212,193554189,193554189,91563661.0,"29,95",503,503-Steunpallet:,484779154
98,133376,287620,282430,451,183178.0,0,0,0,0,3,19/05/2023 16:00:00,19/05/2023 21:00:00,652659,IMPERIAL BISTRO TAFEL GE �60CM,039054075A0,DCONS08,User1428,DP1,1023010779,soares paulo,WP,NORMD60,PCK,9196212,193554190,193554190,91563661.0,"39,95",503,503-Steunpallet:,484779155


In [9]:
path = '/Volumes/2/casa/data/salesstock$vw-12_2019.csv'
df = pd.read_csv(path, low_memory=False, sep=';')
df.head(10)

,ARTICLEID,LOCATIONID,PICKLOACTIONID,TASKATTRIBUTEID,DSPWAVEID,ORIGINALQUANTITY,QUANTITY,VOLUME,WEIGHT,SECONDSSPENT,ARTNR,ARTDESCR,LOCNAME,RES,USERNAME,TEAMNAME,RECIPIENT,RCPNAME,ORDTYPE,ORDARTNORMTIMEGROUP,TRIGGERMOMENT,PALNUMBER,CLAIMID,PICKCLAIMID,TASKID,PRICE,CONTTYPE,CTNAME,SALESSTOCKID,TIMESTAMP
0,103459,3519,151781,306,152820.0,1,1,"0,0315375","1,516",0,641116,KARI HANGLAMP E27,Z-02EXPL01,NaN,User1884,DD,0250,CASA SALON DE PROVENCE,KO,NORMD30,SHP,7724448,173540416,173540416,NaN,"39,95",501,501-Wegwerppallet :,319337575,2/12/2019 17:00:00
1,96628,153578,100809,304,152975.0,1,1,"0,03168","5,98",0,588798,MATHIAS KINDERTAFEL,DWZSEAL2,D-LAADLOS01,User446,DP1,0147,CASA TIGNIEU CC LECLERC,KO,NORMD30,DSP,7733656,173705109,173705109,80998478.0,"39,95",501,501-Wegwerppallet :,320185548,5/12/2019 12:00:00
2,100032,115251,115251,300,152975.0,12,12,"0,0055545","2,043",0,640563,ALU KANDELAAR GOUD H12CM,03404603200,DORPV08,User1671,DP1,0147,CASA TIGNIEU CC LECLERC,KO,NORMD40,PCK,7733671,173705228,173705228,80994335.0,"3,95",501,501-Wegwerppallet :,320234081,5/12/2019 13:00:00
3,77720,117054,148727,306,152960.0,18,18,"0,0146205","4,98",0,607467,AUTHENTIC GLAS 25CL,DPRT43,D-LAADLOS07,User459,DP2,0595,CASA BARCELONA DIAGONALMAR,KO,NORMD30,EXP,7732658,173687968,173687968,NaN,"1,95",501,501-Wegwerppallet :,320264338,5/12/2019 15:00:00
4,77720,117054,148727,306,152960.0,2,2,"0,0016245","0,553333333333",0,607467,AUTHENTIC GLAS 25CL,DPRT43,D-LAADLOS07,User459,DP2,0595,CASA BARCELONA DIAGONALMAR,KO,NORMD30,EXP,7732658,173689251,173689251,NaN,"1,95",501,501-Wegwerppallet :,320264366,5/12/2019 15:00:00
5,103930,117054,112332,306,152960.0,1,1,"0,0015","0,434166666667",0,574084,NEW RESIN ZEEPDISPENSER,DPRT43,D-LAADLOS07,User459,DP2,0595,CASA BARCELONA DIAGONALMAR,KO,NORMD40,EXP,7732667,173687901,173687901,NaN,"6,95",501,501-Wegwerppallet :,320264655,5/12/2019 15:00:00
6,100032,240467,115251,301,152975.0,0,0,0,0,0,640563,ALU KANDELAAR GOUD H12CM,DORPV52,DORPV52,User397,DP2,0147,CASA TIGNIEU CC LECLERC,KO,NORMD40,DSP,7733671,173705228,173705228,81001184.0,"3,95",501,501-Wegwerppallet :,320244623,5/12/2019 14:00:00
7,100033,240467,111867,301,152975.0,0,0,0,0,0,640570,ALU KANDELAAR GOUD H16CM,DORPV52,DORPV52,User397,DP2,0147,CASA TIGNIEU CC LECLERC,KO,NORMD40,DSP,7733671,173705230,173705230,81001184.0,"4,95",501,501-Wegwerppallet :,320244625,5/12/2019 14:00:00
8,100032,240467,115251,301,152975.0,0,0,0,0,0,640563,ALU KANDELAAR GOUD H12CM,DORPV52,DORPV52,User397,DP2,0147,CASA TIGNIEU CC LECLERC,KO,NORMD40,DSP,7733671,173705228,173705228,81001184.0,"3,95",501,501-Wegwerppallet :,320244751,5/12/2019 14:00:00
9,100033,240467,111867,301,152975.0,0,0,0,0,0,640570,ALU KANDELAAR GOUD H16CM,DORPV52,DORPV52,User397,DP2,0147,CASA TIGNIEU CC LECLERC,KO,NORMD40,DSP,7733671,173705230,173705230,81001184.0,"4,95",501,501-Wegwerppallet :,320244753,5/12/2019 14:00:00
